In [2]:
import os, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np, kagglehub
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms, models
from aijack.defense import GeneralMomentAccountant, DPSGDManager

# --- PARAMETRI ---
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
batch_size = 32
epochs = 10
lr = 0.0005

# Parametri Privacy (DP-SGD)
sigma = 0.8        # Rumore gaussiano
l2_norm_clip = 1.0 # Clipping del gradiente
delta = 1e-5       # Parametro di fallimento privacy

# --- DOWNLOAD E PREPARAZIONE DATI ---
path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database")
data_dir = os.path.join(path, 'COVID-19_Radiography_Dataset')

transform = transforms.Compose([
    transforms.Resize((128, 128)), # Risoluzione maggiore per classificazione
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Ispezione rapida classi
print(f"Classi rilevate: {full_dataset.classes}")
# Suddivisione in 3 client (Federated Learning)
indices = np.random.permutation(len(full_dataset))
split = len(full_dataset) // 3
client_loaders = [DataLoader(Subset(full_dataset, indices[i*split : (i+1)*split]), 
                  batch_size=batch_size, shuffle=True) for i in range(3)]

print(f"🚀 Sistema pronto su: {device}")

/Users/flaviafuscaldi/Desktop/fl_thesis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Classi rilevate: ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
🚀 Sistema pronto su: mps


In [3]:
def get_classifier(num_classes):
    # Carichiamo ResNet18 con i pesi di ImageNet
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # Cambiamo il primo strato per accettare 1 canale (Grayscale)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    # Cambiamo l'ultimo strato per le tue 4 classi: COVID, Opacity, Normal, Pneumonia
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)

global_model = get_classifier(len(full_dataset.classes))

In [4]:
# Setup dell'Accountant e del Manager per DP-SGD
accountant = GeneralMomentAccountant(noise_type="Gaussian", backend="python")

privacy_manager = DPSGDManager(
    accountant,
    optim.Adam,
    l2_norm_clip=l2_norm_clip,
    dataset=full_dataset,
    lot_size=batch_size,
    batch_size=batch_size,
    iterations=epochs * (len(full_dataset) // batch_size)
)

# Privatizzazione dell'ottimizzatore e creazione dei loader speciali
dpoptimizer_cls, lot_loader, batch_loader = privacy_manager.privatize(noise_multiplier=sigma)
optimizer = dpoptimizer_cls(global_model.parameters(), lr=lr)

# --- FIX NECESSARIO: Inizializzazione dei gradienti per AIJack ---
dummy_x, _ = next(iter(client_loaders[0]))
global_model(dummy_x.to(device)).mean().backward()
optimizer.zero_grad()

print("🛡️ Privacy Manager configurato. Pronto per l'addestramento.")

🛡️ Privacy Manager configurato. Pronto per l'addestramento.


In [5]:
from tqdm import tqdm
criterion = nn.CrossEntropyLoss()

print("🏋️ Inizio Training Federato Privato (4 Classi)...")
for epoch in range(epochs):
    correct = 0
    total = 0
    # Loop sui "Lotti" (Lot-based DP-SGD)
    pbar = tqdm(lot_loader(optimizer), desc=f"Epoca {epoch+1}")
    
    for X_lot, y_lot in pbar:
        # Loop sui "Batch" interni
        for X_batch, y_batch in batch_loader(TensorDataset(X_lot, y_lot)):
            optimizer.zero_grad()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            outputs = global_model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            # Calcolo metriche per la tesi
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
            
        # Monitoraggio Epsilon in tempo reale
        current_eps = accountant.get_epsilon(delta=delta)
        pbar.set_postfix({"Acc": f"{100.*correct/total:.2f}%", "eps": f"{current_eps:.2f}"})

    print(f"✅ Epoca {epoch+1} conclusa | Accuratezza Finale: {100.*correct/total:.2f}%")

🏋️ Inizio Training Federato Privato (4 Classi)...


Epoca 1:  25%|██▌       | 3350/13220 [11:32<33:59,  4.84it/s, Acc=46.06%, eps=1.14]  


KeyboardInterrupt: 

In [6]:
def get_fast_classifier(num_classes):
    # Carichiamo la ResNet18 con i pesi pre-addestrati
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # 1. CONGELIAMO tutti i parametri: non calcoleremo i gradienti per questi
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. SBLOCCHIAMO solo il primo strato (per gestire le immagini in scala di grigi)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    for param in model.conv1.parameters():
        param.requires_grad = True
    
    # 3. SBLOCCHIAMO l'ultimo strato (per imparare a classificare le 4 classi)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    for param in model.fc.parameters():
        param.requires_grad = True
    
    return model.to(device)

global_model = get_fast_classifier(len(full_dataset.classes))

In [7]:
# Passiamo solo i parametri che hanno 'requires_grad = True'
trainable_params = filter(lambda p: p.requires_grad, global_model.parameters())
optimizer = dpoptimizer_cls(trainable_params, lr=lr)

# Inizializzazione gradienti (obbligatoria per AIJack)
dummy_x, _ = next(iter(client_loaders[0]))
global_model(dummy_x.to(device)).mean().backward()
optimizer.zero_grad()

print("⚡ Modello ottimizzato per la velocità pronto.")

⚡ Modello ottimizzato per la velocità pronto.


In [8]:
from tqdm import tqdm
import time

criterion = nn.CrossEntropyLoss()
print(f"🏋️ Inizio Training Veloce su {device}...")

for epoch in range(epochs):
    correct, total = 0, 0
    start_time = time.time()
    
    # La barra di progressione mostrerà il tempo rimanente
    pbar = tqdm(lot_loader(optimizer), desc=f"Epoca {epoch+1}/{epochs}", unit="lot")
    
    for X_lot, y_lot in pbar:
        for X_batch, y_batch in batch_loader(TensorDataset(X_lot, y_lot)):
            optimizer.zero_grad()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            outputs = global_model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            # Calcolo metriche
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
            
        # Aggiornamento statistiche sulla barra
        current_acc = 100. * correct / total
        current_eps = accountant.get_epsilon(delta=delta)
        pbar.set_postfix({
            "Acc": f"{current_acc:.1f}%", 
            "eps": f"{current_eps:.2f}"
        })

    epoch_time = time.time() - start_time
    print(f"✅ Epoca {epoch+1} completata in {epoch_time:.1f}s | Acc: {current_acc:.2f}% | Epsilon: {current_eps:.2f}")

🏋️ Inizio Training Veloce su mps...


Epoca 1/10:  64%|██████▍   | 8526/13220 [1:04:04<35:16,  2.22lot/s, Acc=49.0%, eps=1.26]


KeyboardInterrupt: 